# Backtesting Trading Strategies

This notebook demonstrates how to backtest trading strategies using historical data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime

print("Libraries imported successfully!")

## Load Historical Data

In [ ]:
symbol = "AAPL"
ticker = yf.Ticker(symbol)
data = ticker.history(period="2y")

print(f"Loaded {len(data)} days of data")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")

## Simple Moving Average Crossover Strategy

In [ ]:
def sma_crossover_backtest(data, short_window=50, long_window=200, initial_capital=10000):
    """
    Backtest SMA crossover strategy
    """
    # Calculate moving averages
    data['SMA_Short'] = data['Close'].rolling(window=short_window).mean()
    data['SMA_Long'] = data['Close'].rolling(window=long_window).mean()
    
    # Generate signals
    data['Signal'] = 0
    data['Signal'][short_window:] = np.where(
        data['SMA_Short'][short_window:] > data['SMA_Long'][short_window:], 1, 0
    )
    
    # Generate positions
    data['Position'] = data['Signal'].diff()
    
    # Calculate returns
    data['Returns'] = data['Close'].pct_change()
    data['Strategy_Returns'] = data['Returns'] * data['Signal'].shift(1)
    
    # Calculate cumulative returns
    data['Cumulative_Returns'] = (1 + data['Returns']).cumprod()
    data['Cumulative_Strategy'] = (1 + data['Strategy_Returns']).cumprod()
    
    # Calculate portfolio values
    data['Portfolio_Value'] = initial_capital * data['Cumulative_Strategy']
    data['Buy_Hold_Value'] = initial_capital * data['Cumulative_Returns']
    
    return data

# Run backtest
results = sma_crossover_backtest(data.copy(), short_window=50, long_window=200)

# Calculate metrics
total_return = (results['Portfolio_Value'].iloc[-1] / 10000 - 1) * 100
buy_hold_return = (results['Buy_Hold_Value'].iloc[-1] / 10000 - 1) * 100
buy_signals = len(results[results['Position'] == 1])
sell_signals = len(results[results['Position'] == -1])

print("=== Backtest Results ===")
print(f"Strategy Return: {total_return:.2f}%")
print(f"Buy & Hold Return: {buy_hold_return:.2f}%")
print(f"Buy Signals: {buy_signals}")
print(f"Sell Signals: {sell_signals}")
print(f"Final Portfolio Value: ${results['Portfolio_Value'].iloc[-1]:.2f}")

## Visualize Strategy Performance

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Price and Moving Averages
axes[0].plot(results.index, results['Close'], label='Close Price', linewidth=2)
axes[0].plot(results.index, results['SMA_Short'], label='SMA 50', alpha=0.7)
axes[0].plot(results.index, results['SMA_Long'], label='SMA 200', alpha=0.7)
axes[0].scatter(
    results[results['Position'] == 1].index,
    results[results['Position'] == 1]['Close'],
    color='green', marker='^', s=100, label='Buy Signal', zorder=5
)
axes[0].scatter(
    results[results['Position'] == -1].index,
    results[results['Position'] == -1]['Close'],
    color='red', marker='v', s=100, label='Sell Signal', zorder=5
)
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'{symbol} - SMA Crossover Strategy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Portfolio Value Comparison
axes[1].plot(results.index, results['Portfolio_Value'], label='Strategy', linewidth=2)
axes[1].plot(results.index, results['Buy_Hold_Value'], label='Buy & Hold', linewidth=2, linestyle='--')
axes[1].axhline(y=10000, color='gray', linestyle=':', alpha=0.5, label='Initial Capital')
axes[1].set_ylabel('Portfolio Value ($)')
axes[1].set_title('Portfolio Performance Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Cumulative Returns
axes[2].plot(results.index, (results['Cumulative_Strategy'] - 1) * 100, label='Strategy', linewidth=2)
axes[2].plot(results.index, (results['Cumulative_Returns'] - 1) * 100, label='Buy & Hold', linewidth=2, linestyle='--')
axes[2].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_ylabel('Cumulative Returns (%)')
axes[2].set_xlabel('Date')
axes[2].set_title('Cumulative Returns Comparison')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Calculate Additional Metrics

In [ ]:
# Calculate Sharpe Ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0.02):
    """
    Calculate annualized Sharpe ratio
    """
    excess_returns = returns - (risk_free_rate / 252)  # Daily risk-free rate
    if excess_returns.std() == 0:
        return 0
    sharpe = np.sqrt(252) * (excess_returns.mean() / excess_returns.std())
    return sharpe

# Calculate metrics
strategy_returns = results['Strategy_Returns'].dropna()
buy_hold_returns = results['Returns'].dropna()

strategy_sharpe = calculate_sharpe_ratio(strategy_returns)
buy_hold_sharpe = calculate_sharpe_ratio(buy_hold_returns)

strategy_volatility = strategy_returns.std() * np.sqrt(252) * 100
buy_hold_volatility = buy_hold_returns.std() * np.sqrt(252) * 100

print("=== Risk Metrics ===")
print(f"Strategy Sharpe Ratio: {strategy_sharpe:.2f}")
print(f"Buy & Hold Sharpe Ratio: {buy_hold_sharpe:.2f}")
print(f"Strategy Volatility: {strategy_volatility:.2f}%")
print(f"Buy & Hold Volatility: {buy_hold_volatility:.2f}%")

## Drawdown Analysis

In [ ]:
# Calculate drawdown
def calculate_drawdown(portfolio_values):
    """
    Calculate drawdown series
    """
    peak = portfolio_values.expanding().max()
    drawdown = (portfolio_values - peak) / peak * 100
    return drawdown

results['Strategy_Drawdown'] = calculate_drawdown(results['Portfolio_Value'])
results['Buy_Hold_Drawdown'] = calculate_drawdown(results['Buy_Hold_Value'])

# Plot drawdown
plt.figure(figsize=(14, 6))
plt.fill_between(results.index, results['Strategy_Drawdown'], 0, alpha=0.3, label='Strategy')
plt.fill_between(results.index, results['Buy_Hold_Drawdown'], 0, alpha=0.3, label='Buy & Hold')
plt.plot(results.index, results['Strategy_Drawdown'], linewidth=1, label='Strategy')
plt.plot(results.index, results['Buy_Hold_Drawdown'], linewidth=1, label='Buy & Hold', linestyle='--')
plt.ylabel('Drawdown (%)')
plt.xlabel('Date')
plt.title('Drawdown Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

max_drawdown_strategy = results['Strategy_Drawdown'].min()
max_drawdown_buyhold = results['Buy_Hold_Drawdown'].min()

print(f"Maximum Drawdown - Strategy: {max_drawdown_strategy:.2f}%")
print(f"Maximum Drawdown - Buy & Hold: {max_drawdown_buyhold:.2f}%")